***
### Import of required libraries
***

In [1]:
import warnings
import plotly

warnings.simplefilter(action="ignore", category=FutureWarning)
plotly.io.kaleido.scope.mathjax = None

import os
from pathlib import Path
import sys
import numpy as np
from dataclasses import dataclass, field
from collections import defaultdict
from math import cos, radians
import datetime as dt

from traffic.core import Traffic
from traffic.data import navaids
from traffic.data import eurofirs
import pandas as pd
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
import matplotlib.patheffects as pe
import plotly.graph_objects as go
import plotly.express as px
from cartes.crs import EuroPP
import cartopy.feature as cfeature
import cartopy.io.shapereader as shpreader
from statsmodels.stats.proportion import proportions_ztest
from scipy.stats import mannwhitneyu, ttest_ind
import onnxruntime as ort

In [4]:
try:
    from helper import aligned_navpoint_, add_fuelflow
except ModuleNotFoundError:
    # relative import didn't work, adding the path manually
    pwd = os.getcwd()
    helper_dir = os.path.join(pwd, "git", "RAD_inefficiencies_paper", "analysis")
    sys.path.append(helper_dir)
    from helper import aligned_navpoint_, add_fuelflow

***
### Define some useful functions
***

In [41]:
def plot_overview(traf, title, save_as=None, overview=False):
    try:
        # real ugly hack to solve the issue with incompatible versions of basemap and and numpy
        # I need a more or less recent version of numpy (to unpickle data), but basemap is a bit outdated
        from mpl_toolkits.basemap import Basemap
    except ImportError:
        return

    if save_as:
        # suppress displaying the plot unless you call show()
        plt.ioff()

    font_and_marker_color = "black"
    font_background_color = "white"

    if overview:
        fig, ax = plt.subplots(
            1,
            figsize=(20, 20),
        )
        llcrnrlon = -1.2
        urcrnrlon = 9
        llcrnrlat = 46.4
        urcrnrlat = 52.1
    else:
        fig, ax = plt.subplots(
            1,
            figsize=(15, 5),
        )
        llcrnrlon = 3.5
        urcrnrlon = 8.6
        llcrnrlat = 47
        urcrnrlat = 48.5

    m = Basemap(
        llcrnrlon=llcrnrlon,
        urcrnrlon=urcrnrlon,
        llcrnrlat=llcrnrlat,
        urcrnrlat=urcrnrlat,
        resolution="i",
        epsg="3035",
        lat_ts=llcrnrlat,
        lat_0=(llcrnrlat + urcrnrlat) / 2,
        lon_0=(llcrnrlon + urcrnrlon) / 2,
        ax=ax,
    )

    m.drawcoastlines(linewidth=0.3)
    m.drawcountries(linewidth=0.3)
    img = m.arcgisimage(service="World_Shaded_Relief", xpixels=2000, verbose=False)
    img.set_alpha(0.4)

    for flight in traf:
        x, y = m(flight.data.longitude, flight.data.latitude)
        ax.scatter(
            x,
            y,
            color="mediumblue",
            s=0.001,
            alpha=0.1,
        )

    # Navaids
    navaids_list = ["LUMEL", "TORPA", "REKLA", "RLP"]
    for navaid in navaids_list:
        x, y = m(navaids[navaid].longitude, navaids[navaid].latitude)
        ax.scatter(
            x,
            y,
            color=font_and_marker_color,
            s=100,
        )
        ax.annotate(
            navaid,
            xy=(x, y),
            xytext=(x - 1e4, y - 1e4),
            fontsize=14,
            color=font_and_marker_color,
            path_effects=[pe.withStroke(linewidth=3, foreground=font_background_color)],
        )

    if save_as:
        fig.savefig(save_as, bbox_inches="tight")
        plt.ion()
    else:
        fig.show()
    plt.close(fig)

In [6]:
def plot_sample(df, n=10, head=True):
    df_dist = (
        df.groupby("flight_id")["cumdist"]
        .max()
        .reset_index()
        .sort_values("cumdist", ascending=False)
    )
    if head:
        flight_ids = df_dist["flight_id"].head(n).to_list()
    else:
        flight_ids = df_dist["flight_id"].tail(n).to_list()

    fig = go.Figure()
    for flight_id in flight_ids:
        name = f"{flight_id}, {df_dist.query('flight_id == @flight_id').cumdist.values[0]:.0f}NM"
        fig.add_trace(
            go.Scattermapbox(
                lat=df.query("flight_id == @flight_id").latitude,
                lon=df.query("flight_id == @flight_id").longitude,
                mode="markers+lines",
                name=name,
            )
        )

    fig.update_layout(
        width=1200,
        height=800,
        margin=dict(l=50, r=0, t=40, b=40),
        mapbox=dict(
            style="carto-positron",
        ),
    )

    fig.show()

In [7]:
def output_flight_plan_stats(all_ids, routed_ids):
    print(f"Flight plans filed: {len(all_ids)}")
    print(f"Flight plans filed via LUMEL TORPA REKLA RLP: {len(routed_ids)}")
    print(f"Matched {len(routed_ids) / len(all_ids) * 100:.2f}% of filed flightplans")

In [8]:
def output_flights_stats(t_all, t_fpln, t_aligned, waypoint):
    print(f"Total trajectories, select operators, all routes: {len(t_all)}")
    print(f"Trajectories filed via LUMEL TORPA REKLA RLP: {len(t_fpln)}")
    print(f"Matched {len(t_fpln) / len(t_all) * 100:.2f}% of filed flightplans")
    print(
        f"Number of flights aligned on {waypoint}: {len(t_aligned)}, out of {len(t_fpln)}, {len(t_aligned) / len(t_fpln) * 100:.2f}%"
    )


In [9]:
def filter_outlier_traj(df):
    ids_to_keep = (
        df.groupby("flight_id")["cumdist"]
        .max()
        .reset_index()
        .sort_values("cumdist", ascending=False)
        .query("420 < cumdist < 1000")
        .flight_id.to_list()
    )
    return df[ids_to_keep]

In [10]:
@dataclass
class FlightData:
    flt_pln: dict = field(default_factory=dict)
    flt_ids: dict = field(default_factory=dict)
    traf: dict = field(default_factory=dict)


data = FlightData()

***
### Load data
***

In [11]:
verbose = False

# configure base paths for data
data_path = "/store/Projects_CRM/RAD_paper/"

flight_plans_paths = [
    os.path.join(data_path, "data_052023_to_052024", "flightplans"),
    os.path.join(data_path, "data_052024_to_092024", "flightplans"),
]

traj_data_paths = [
    os.path.join(data_path, "data_052023_to_052024", "trajectories"),
    os.path.join(data_path, "data_052024_to_092024", "trajectories"),
]


In [12]:
# load flight plans
data.flt_pln["all_flt"] = pd.DataFrame()
data.flt_pln["routed_flt"] = pd.DataFrame()

for flight_plans_path in flight_plans_paths:
    flightplans = pd.read_parquet(
        os.path.join(flight_plans_path, "flightplans_complete_with_flight_id.parquet")
    )
    flightplans["flight_id"] = flightplans["flight_id"].apply(lambda x: x[0])

    data.flt_pln["all_flt"] = pd.concat([data.flt_pln["all_flt"], flightplans])

    flightplans = flightplans[flightplans.origin == "LSZH"]

    filed_REKLA = []
    for id in tqdm(flightplans.identifier.unique()):
        flightplan = flightplans[flightplans.identifier == id]
        if (
            flightplan.segment_id.str.contains("LUMEL").any()
            and flightplan.segment_id.str.contains("TORPA").any()
            and flightplan.segment_id.str.contains("REKLA").any()
            and flightplan.segment_id.str.contains("RLP").any()
        ):
            filed_REKLA.append(id)

    flightplans = flightplans[flightplans.identifier.isin(filed_REKLA)]
    data.flt_pln["routed_flt"] = pd.concat([data.flt_pln["routed_flt"], flightplans])

del flightplans

  0%|          | 0/3443 [00:00<?, ?it/s]

  0%|          | 0/1425 [00:00<?, ?it/s]

In [13]:
# data.flt_ids["routed_swr"] = data.flt_pln["routed_flt"][
#     data.flt_pln["routed_flt"].operator == "SWR"
# ].flight_id.unique()
# data.flt_ids["routed_baw"] = data.flt_pln["routed_flt"][
#     data.flt_pln["routed_flt"].operator == "BAW"
# ].flight_id.unique()
# data.flt_ids["routed_ezy"] = data.flt_pln["routed_flt"][
#     data.flt_pln["routed_flt"].operator == "EZY"
# ].flight_id.unique()

# flights that are planed according to the route LUMEL TORPA REKLA RLP
operators = ["SWR", "BAW", "EZY"]
data.flt_ids["routed_egll"] = (
    data.flt_pln["routed_flt"]
    .query("origin == 'LSZH' and destination == 'EGLL' and operator in @operators")
    .flight_id.unique()
)
data.flt_ids["routed_egkk"] = (
    data.flt_pln["routed_flt"]
    .query("origin == 'LSZH' and destination == 'EGKK' and operator in @operators")
    .flight_id.unique()
)
data.flt_ids["routed_eglc"] = (
    data.flt_pln["routed_flt"]
    .query("origin == 'LSZH' and destination == 'EGLC' and operator in @operators")
    .flight_id.unique()
)

In [14]:
# load trajectories
data.traf["egll_all"] = None
data.traf["egkk_all"] = None
data.traf["eglc_all"] = None

for traj_data_path in traj_data_paths:
    if data.traf["egll_all"] is None:
        data.traf["egll_all"] = (
            Traffic.from_file(
                os.path.join(traj_data_path, "zurich_to_london_EGLL.parquet")
            )
            .drop(columns=["serials"])
            .aircraft_data()
        )  # Heathrow
    else:
        data.traf["egll_all"] = (
            data.traf["egll_all"]
            + Traffic.from_file(
                os.path.join(traj_data_path, "zurich_to_london_EGLL.parquet")
            )
            .drop(columns=["serials"])
            .aircraft_data()
        )

    if data.traf["egkk_all"] is None:
        data.traf["egkk_all"] = (
            Traffic.from_file(
                os.path.join(traj_data_path, "zurich_to_london_EGKK.parquet")
            )
            .drop(columns=["serials"])
            .aircraft_data()
        )  # Gatwick
    else:
        data.traf["egkk_all"] = (
            data.traf["egkk_all"]
            + Traffic.from_file(
                os.path.join(traj_data_path, "zurich_to_london_EGKK.parquet")
            )
            .drop(columns=["serials"])
            .aircraft_data()
        )

    if data.traf["eglc_all"] is None:
        data.traf["eglc_all"] = (
            Traffic.from_file(
                os.path.join(traj_data_path, "zurich_to_london_EGLC.parquet")
            )
            .drop(columns=["serials"])
            .aircraft_data()
        )  # City
    else:
        data.traf["eglc_all"] = (
            data.traf["eglc_all"]
            + Traffic.from_file(
                os.path.join(traj_data_path, "zurich_to_london_EGLC.parquet")
            )
            .drop(columns=["serials"])
            .aircraft_data()
        )

data.traf["all"] = data.traf["egll_all"] + data.traf["egkk_all"] + data.traf["eglc_all"]


download:   0%|          | 0/92295 [00:00<?, ?it/s]

***
### Calculate various things and output/print them
***

In [15]:
# add cumulative distance to the trajectories
data.traf["egll_all"] = (
    data.traf["egll_all"]
    .airborne()
    .resample("5s")
    .cumulative_distance(compute_gs=False, compute_track=False)
    .eval(desc="cumulative_distance", max_workers=20)
)
data.traf["egkk_all"] = (
    data.traf["egkk_all"]
    .airborne()
    .resample("5s")
    .cumulative_distance(compute_gs=False, compute_track=False)
    .eval(desc="cumulative_distance", max_workers=20)
)
data.traf["eglc_all"] = (
    data.traf["eglc_all"]
    .airborne()
    .resample("5s")
    .cumulative_distance(compute_gs=False, compute_track=False)
    .eval(desc="cumulative_distance", max_workers=20)
)
data.traf["all"] = (
    data.traf["all"]
    .airborne()
    .resample("5s")
    .cumulative_distance(compute_gs=False, compute_track=False)
    .eval(desc="cumulative_distance", max_workers=20)
)

cumulative_distance:   0%|          | 0/4461 [00:00<?, ?it/s]

cumulative_distance:   0%|          | 0/1078 [00:00<?, ?it/s]

cumulative_distance:   0%|          | 0/6790 [00:00<?, ?it/s]

cumulative_distance:   0%|          | 0/12329 [00:00<?, ?it/s]

In [16]:
# remove outliers from the trajectories
data.traf["egll_all"] = filter_outlier_traj(data.traf["egll_all"])
data.traf["egkk_all"] = filter_outlier_traj(data.traf["egkk_all"])
data.traf["eglc_all"] = filter_outlier_traj(data.traf["eglc_all"])
data.traf["all"] = filter_outlier_traj(data.traf["all"])

In [17]:
# data.traf["swr"] = data.traf["all"][data.flt_ids["routed_swr"]]
# data.traf["baw"] = data.traf["all"][data.flt_ids["routed_baw"]]
# data.traf["ezy"] = data.traf["all"][data.flt_ids["routed_ezy"]]

# trajectories filed via LUMEL TORPA REKLA RLP
data.traf["routed_egll"] = data.traf["egll_all"][data.flt_ids["routed_egll"]]
data.traf["routed_egkk"] = data.traf["egkk_all"][data.flt_ids["routed_egkk"]]
data.traf["routed_eglc"] = data.traf["eglc_all"][data.flt_ids["routed_eglc"]]


In [18]:
if verbose:
    plot_sample(data.traf["routed_egll"].data, n=10, head=True)

In [19]:
# find flights via REKLA
data.traf["routed_egll_rekla"] = (
    data.traf["routed_egll"]
    .iterate_lazy()
    .pipe(aligned_navpoint_, "REKLA", angle_precision=10, min_distance=10)
    .eval(desc="Aligned on REKLA", max_workers=20)
)

data.traf["routed_egkk_rekla"] = (
    data.traf["routed_egkk"]
    .iterate_lazy()
    .pipe(aligned_navpoint_, "REKLA", angle_precision=10, min_distance=10)
    .eval(desc="Aligned on REKLA", max_workers=20)
)

data.traf["routed_eglc_rekla"] = (
    data.traf["routed_eglc"]
    .iterate_lazy()
    .pipe(aligned_navpoint_, "REKLA", angle_precision=10, min_distance=10)
    .eval(desc="Aligned on REKLA", max_workers=20)
)

Aligned on REKLA:   0%|          | 0/2439 [00:00<?, ?it/s]

Aligned on REKLA:   0%|          | 0/585 [00:00<?, ?it/s]

Aligned on REKLA:   0%|          | 0/807 [00:00<?, ?it/s]

In [20]:
# df_dist = (
#     data.traf["routed_egll_rekla"]
#     .data.groupby("flight_id")["cumdist"]
#     .max()
#     .reset_index()
#     .sort_values("cumdist", ascending=False)
# )
# # print(f"Median distance for REKLA: {df_dist['cumdist'].median()}")
# if verbose:
#     df_dist

In [21]:
# trajectories of Swiss, EazyJet and British Airways
data.traf["egll_ops"] = data.traf["egll_all"].query(
    "callsign.str.startswith('SWR') or callsign.str.startswith('BAW') or callsign.str.startswith('EZY')"
)
data.traf["egkk_ops"] = data.traf["egkk_all"].query(
    "callsign.str.startswith('SWR') or callsign.str.startswith('BAW') or callsign.str.startswith('EZY')"
)
data.traf["eglc_ops"] = data.traf["eglc_all"].query(
    "callsign.str.startswith('SWR') or callsign.str.startswith('BAW') or callsign.str.startswith('EZY')"
)


In [22]:
# all flights by the operators to London
data.flt_pln["egll_flt"] = (
    data.flt_pln["all_flt"]
    .query("origin == 'LSZH' and destination == 'EGLL' and operator in @operators")
    .flight_id.unique()
)
data.flt_pln["egkk_flt"] = (
    data.flt_pln["all_flt"]
    .query("origin == 'LSZH' and destination == 'EGKK' and operator in @operators")
    .flight_id.unique()
)
data.flt_pln["eglc_flt"] = (
    data.flt_pln["all_flt"]
    .query("origin == 'LSZH' and destination == 'EGLC' and operator in @operators")
    .flight_id.unique()
)


In [23]:
# output some statistics on the matching
print("EGLL")
output_flight_plan_stats(
    all_ids=data.flt_pln["egll_flt"], routed_ids=data.flt_ids["routed_egll"]
)
print("\n")
output_flights_stats(
    t_all=data.traf["egll_ops"],
    t_fpln=data.traf["routed_egll"],
    t_aligned=data.traf["routed_egll_rekla"],
    waypoint="REKLA",
)
print(
    f"{len(data.traf['routed_egll']) / len(data.flt_ids['routed_egll']) * 100:.1f}% match of traj and flighplans (via waypoints)"
)
print("\n\n")

print("EGKK")
output_flight_plan_stats(
    all_ids=data.flt_pln["egkk_flt"], routed_ids=data.flt_ids["routed_egkk"]
)
print("\n")
output_flights_stats(
    t_all=data.traf["egkk_ops"],
    t_fpln=data.traf["routed_egkk"],
    t_aligned=data.traf["routed_egkk_rekla"],
    waypoint="REKLA",
)
print(
    f"{len(data.traf['routed_egkk']) / len(data.flt_ids['routed_egkk']) * 100:.1f}% match of traj and flighplans (via waypoints)"
)
print("\n\n")

print("EGLC")
output_flight_plan_stats(
    all_ids=data.flt_pln["eglc_flt"], routed_ids=data.flt_ids["routed_eglc"]
)
print("\n")
output_flights_stats(
    t_all=data.traf["eglc_ops"],
    t_fpln=data.traf["routed_eglc"],
    t_aligned=data.traf["routed_eglc_rekla"],
    waypoint="REKLA",
)
print(
    f"{len(data.traf['routed_eglc']) / len(data.flt_ids['routed_eglc']) * 100:.1f}% match of traj and flighplans (via waypoints)"
)


EGLL
Flight plans filed: 3138
Flight plans filed via LUMEL TORPA REKLA RLP: 2953
Matched 94.10% of filed flightplans


Total trajectories, select operators, all routes: 2741
Trajectories filed via LUMEL TORPA REKLA RLP: 2439
Matched 88.98% of filed flightplans
Number of flights aligned on REKLA: 110, out of 2439, 4.51%
82.6% match of traj and flighplans (via waypoints)



EGKK
Flight plans filed: 680
Flight plans filed via LUMEL TORPA REKLA RLP: 653
Matched 96.03% of filed flightplans


Total trajectories, select operators, all routes: 687
Trajectories filed via LUMEL TORPA REKLA RLP: 585
Matched 85.15% of filed flightplans
Number of flights aligned on REKLA: 26, out of 585, 4.44%
89.6% match of traj and flighplans (via waypoints)



EGLC
Flight plans filed: 1036
Flight plans filed via LUMEL TORPA REKLA RLP: 894
Matched 86.29% of filed flightplans


Total trajectories, select operators, all routes: 966
Trajectories filed via LUMEL TORPA REKLA RLP: 807
Matched 83.54% of filed flightplan

In [24]:
# Fly As Filed, statistical tests on the different groups (by destination)
destinations = ["egll", "egkk", "eglc"]
count_flown_before = 0
count_flown_after = 0
count_filed_before = 0
count_filed_after = 0
for dest in destinations:
    before_filed = data.traf[f"routed_{dest}"].query(
        "timestamp < '2024-05-01 00:00:00Z'"
    )
    before_flown = data.traf[f"routed_{dest}_rekla"].query(
        "timestamp < '2024-05-01 00:00:00Z'"
    )
    after_filed = data.traf[f"routed_{dest}"].query(
        "timestamp >= '2024-05-01 00:00:00Z'"
    )
    after_flown = data.traf[f"routed_{dest}_rekla"].query(
        "timestamp >= '2024-05-01 00:00:00Z'"
    )

    counts = np.array([len(before_flown), len(after_flown)])
    nobs = np.array([len(before_filed), len(after_filed)])
    stat, pval = proportions_ztest(counts, nobs, alternative="smaller")

    count_flown_before += len(before_flown)
    count_flown_after += len(after_flown)
    count_filed_before += len(before_filed)
    count_filed_after += len(after_filed)

    print("")

    print(f"Destination: {dest.upper()}")
    print(f"Before filed: {len(before_filed)}")
    print(f"Before flown: {len(before_flown)}")
    print(f"Before ratio: {len(before_flown)/len(before_filed)*100:.1f}%")
    print(f"After filed: {len(after_filed)}")
    print(f"After flown: {len(after_flown)}")
    print(f"After ratio: {len(after_flown)/len(after_filed)*100:.1f}%")
    print(f"p-value: {pval:.2f}")
    print("\n")



Destination: EGLL
Before filed: 1689
Before flown: 68
Before ratio: 4.0%
After filed: 752
After flown: 42
After ratio: 5.6%
p-value: 0.04



Destination: EGKK
Before filed: 354
Before flown: 16
Before ratio: 4.5%
After filed: 231
After flown: 10
After ratio: 4.3%
p-value: 0.54



Destination: EGLC
Before filed: 580
Before flown: 18
Before ratio: 3.1%
After filed: 227
After flown: 4
After ratio: 1.8%
p-value: 0.85




In [25]:
# Fly As Filed, statistical tests on the different groups (all destination)
counts = np.array([count_flown_before, count_flown_after])
nobs = np.array([count_filed_before, count_filed_after])
stat, pval = proportions_ztest(counts, nobs)
print(f"p-value: {pval:.4f}")
print(f"statistic: {stat}")
print(f"filed before: {count_filed_before}")
print(f"filed after: {count_filed_after}")
print(f"flown before: {count_flown_before}")
print(f"flown after: {count_flown_after}")
print(f"ratio before: {count_flown_before/count_filed_before*100:.1f}%")
print(f"ratio after: {count_flown_after/count_filed_after*100:.1f}%")

p-value: 0.2845
statistic: -1.0702776182938671
filed before: 2623
filed after: 1210
flown before: 102
flown after: 56
ratio before: 3.9%
ratio after: 4.6%


In [26]:
data.traf["routed_egll_non_rekla"] = (
    data.traf["routed_egll"] - data.traf["routed_egll_rekla"]
)
data.traf["routed_egkk_non_rekla"] = (
    data.traf["routed_egkk"] - data.traf["routed_egkk_rekla"]
)
data.traf["routed_eglc_non_rekla"] = (
    data.traf["routed_eglc"] - data.traf["routed_eglc_rekla"]
)


# data.traf["routed_egll_non_rlp"] = (
#     data.traf["routed_egll"] - data.traf["routed_egll_rlp"]
# )

In [46]:
# produce some plots
plot_overview(
    data.traf["routed_egll_non_rekla"],
    "Flights not via REKLA",
    save_as="egll_non_rekla_combined.png",
    overview=False,
)
plot_overview(
    data.traf["routed_egll"],
    "All flights",
    save_as="egll_zoomed_combined.png",
    overview=False,
)
# all flights
plot_overview(
    data.traf["routed_egll"],
    "All flights",
    save_as="egll_all_combined.png",
    overview=True,
)

In [ ]:
# output statistics and plots about the duration of the flights
destintations = ["egll", "egkk", "eglc"]
for dest in destintations:
    print(f"\n\n{dest.upper()}")
    routed_reklas = f"routed_{dest}_rekla"
    routed_non_reklas = f"routed_{dest}_non_rekla"

    dist_dt_rekla = (
        data.traf[routed_reklas]
        .data.groupby(["flight_id"])["timestamp"]
        .agg(np.ptp)
        .reset_index(name="diff")["diff"]
        .dt.total_seconds()
        / 60
    )

    dist_dt_non_rekla = (
        data.traf[routed_non_reklas]
        .data.groupby(["flight_id"])["timestamp"]
        .agg(np.ptp)
        .reset_index(name="diff")["diff"]
        .dt.total_seconds()
        / 60
    )

    xbins = dict(
        start=65, end=180, size=3
    )

    fig = go.Figure()
    fig.add_trace(
        go.Histogram(
            x=dist_dt_non_rekla,
            histnorm="probability",
            xbins=xbins,
            name="not via REKLA",
        )
    )
    fig.add_trace(
        go.Histogram(
            x=dist_dt_rekla,
            histnorm="probability",
            xbins=xbins,
            name="via REKLA",
        )
    )

    fig.update_layout(
        title=f"{dest.upper()}: Duration",
        xaxis_title="Duration [min]",
        yaxis_title="Probability",
        width=400,
        height=400,
        barmode="overlay",
        legend=dict(yanchor="top", y=0.95, xanchor="left", x=0.45),
        margin=dict(l=0, r=0, t=30, b=0),
        font=dict(size=14),
    )
    fig.update_traces(opacity=0.75)
    fig.update_xaxes(range=[65, 180])
    print(
        f"Median duration in EGTT for REKLA: {dist_dt_rekla.median():.1f} min (flights: {len(dist_dt_rekla)})"
    )
    print(
        f"Median duration in EGTT for non-REKLA: {dist_dt_non_rekla.median():.1f} min (flights: {len(dist_dt_non_rekla)})"
    )
    print(f"Difference: {dist_dt_rekla.median() - dist_dt_non_rekla.median():.1f} min")

    # # two-sided Mann-Whitney U test
    # stat, p = mannwhitneyu(dist_dt_rekla, dist_dt_non_rekla)
    # res = ttest_ind(dist_dt_rekla, dist_dt_non_rekla)

    # one-sided Mann-Whitney U test (first distribution has smaller values)
    stat, p = mannwhitneyu(dist_dt_non_rekla, dist_dt_rekla, alternative="less")
    res = ttest_ind(dist_dt_non_rekla, dist_dt_rekla, alternative="less")

    print(f"Mann-Whitney U test: Statistics={stat:.1f}, p={p:.3f}")
    print(
        f"t-test: Statistics={res.statistic:.1f}, dof={res.df:.1f}, p={res.pvalue:.3f}"
    )

    fig.write_image(f"histogram_rekla_time_{dest}.png")
    fig.show()



EGLL
Median duration in EGTT for REKLA: 88.9 min (flights: 110)
Median duration in EGTT for non-REKLA: 86.5 min (flights: 2329)
Difference: 2.4 min
Mann-Whitney U test: Statistics=106582.5, p=0.001
t-test: Statistics=0.3, dof=2437.0, p=0.620




EGKK
Median duration in EGTT for REKLA: 79.9 min (flights: 26)
Median duration in EGTT for non-REKLA: 81.5 min (flights: 559)
Difference: -1.6 min
Mann-Whitney U test: Statistics=8402.5, p=0.911
t-test: Statistics=1.2, dof=583.0, p=0.879




EGLC
Median duration in EGTT for REKLA: 79.7 min (flights: 22)
Median duration in EGTT for non-REKLA: 78.8 min (flights: 785)
Difference: 0.9 min
Mann-Whitney U test: Statistics=7186.5, p=0.090
t-test: Statistics=-0.9, dof=805.0, p=0.180


In [38]:
# output statistics and plots about the distance of the flights
for dest in destintations:
    print(f"\n\n{dest.upper()}")
    routed_reklas = f"routed_{dest}_rekla"
    routed_non_reklas = f"routed_{dest}_non_rekla"

    dist_rekla = data.traf[routed_reklas].data.groupby("flight_id")["cumdist"].max()
    dist_non_rekla = (
        data.traf[routed_non_reklas].data.groupby("flight_id")["cumdist"].max()
    )

    xbins = dict(
        start=400.0, end=1000.0, size=10
    )

    fig = go.Figure()
    fig.add_trace(
        go.Histogram(
            x=dist_non_rekla,
            histnorm="probability",
            xbins=xbins,
            name="not via REKLA",
        )
    )
    fig.add_trace(
        go.Histogram(
            x=dist_rekla,
            histnorm="probability",
            xbins=xbins,
            name="via REKLA",
        )
    )

    fig.update_layout(
        title=f"{dest.upper()}: Distance",
        xaxis_title="Distance [NM]",
        yaxis_title="Probability",
        width=400,
        height=400,
        barmode="overlay",
        legend=dict(yanchor="top", y=0.95, xanchor="left", x=0.45),
        margin=dict(l=0, r=0, t=30, b=0),
        font=dict(size=14),
    )
    fig.update_traces(opacity=0.75)
    fig.update_xaxes(range=[400, 1000])
    print(f"Median distance in EGTT for REKLA: {dist_rekla.median():.1f} NM")
    print(f"Median distance in EGTT for non-REKLA: {dist_non_rekla.median():.1f} NM")
    print(f"Difference: {dist_rekla.median() - dist_non_rekla.median():.1f} NM")

    # # two-sided Mann-Whitney U test
    # stat, p = mannwhitneyu(dist_dt_rekla, dist_dt_non_rekla)
    # res = ttest_ind(dist_dt_rekla, dist_dt_non_rekla)

    # one-sided Mann-Whitney U test (first distribution has smaller values)
    stat, p = mannwhitneyu(dist_non_rekla, dist_rekla, alternative="less")
    res = ttest_ind(dist_non_rekla, dist_rekla, alternative="less")

    print(f"Mann-Whitney U test: Statistics={stat:.1f}, p={p:.3f}")
    print(
        f"t-test: Statistics={res.statistic:.1f}, dof={res.df:.1f}, p={res.pvalue:.3f}"
    )

    fig.write_image(f"histogram_rekla_dist_{dest}.png")
    fig.show()



EGLL
Median distance in EGTT for REKLA: 494.5 NM
Median distance in EGTT for non-REKLA: 482.5 NM
Difference: 12.0 NM
Mann-Whitney U test: Statistics=82852.0, p=0.000
t-test: Statistics=-4.4, dof=2437.0, p=0.000




EGKK
Median distance in EGTT for REKLA: 454.1 NM
Median distance in EGTT for non-REKLA: 447.3 NM
Difference: 6.7 NM
Mann-Whitney U test: Statistics=5553.0, p=0.021
t-test: Statistics=-0.7, dof=583.0, p=0.233




EGLC
Median distance in EGTT for REKLA: 458.4 NM
Median distance in EGTT for non-REKLA: 447.7 NM
Difference: 10.7 NM
Mann-Whitney U test: Statistics=4088.0, p=0.000
t-test: Statistics=-1.3, dof=805.0, p=0.103


In [51]:
# produce histograms over time for the flights via REKLA
dest = "egll"
start_times = []

for flight in data.traf[f"routed_{dest}_rekla"]:
    start_times.append(flight.start)
print(len(start_times))

df = pd.DataFrame({"start_times": pd.to_datetime(start_times)})
n_bins = (df["start_times"].max() - df["start_times"].min()).days

height = 400
width = 400

day_short = {
    "Monday": "Mon",
    "Tuesday": "Tue",
    "Wednesday": "Wed",
    "Thursday": "Thu",
    "Friday": "Fri",
    "Saturday": "Sat",
    "Sunday": "Sun",
}

df["weekday"] = df["start_times"].dt.day_name().replace(day_short)
df["time"] = df["start_times"].dt.time
df["hour"] = df["start_times"].dt.hour
df["date"] = df["start_times"].dt.date

fig_weekday = px.histogram(
    df,
    x="weekday",
    title="Start Times vs Weekday",
    category_orders={
        "weekday": [
            "Mon",
            "Tue",
            "Wed",
            "Thu",
            "Fri",
            "Sat",
            "Sun",
        ]
    },
)
fig_weekday.update_layout(
    title="Flying as filed, per weekday",
    xaxis_title="Weekday",
    yaxis_title="Count",
    height=height,
    width=width,
    margin=dict(l=0, r=0, t=30, b=0),
    font=dict(size=14),
)

fig_time = go.Figure()
go_time = go.Histogram(x=df["hour"], nbinsx=24, histfunc="count")
fig_time.add_trace(go_time)

fig_time.update_layout(
    title="Flying as filed, per hour of the day",
    xaxis_title="Hour of the day",
    yaxis_title="Count",
    yaxis=dict(
        tickformat="%H:%M"
    ),
    height=height,
    width=width,
    margin=dict(l=0, r=0, t=30, b=0),
    font=dict(size=14),
)

fig_hist = go.Figure()
fig_hist.add_trace(
    go.Histogram(
        x=df["date"],
        nbinsx=n_bins,
    )
)
fig_hist.update_layout(
    title="Flying as filed, per date",
    xaxis_title="Date",
    yaxis_title="Count",
    height=height,
    width=width,
    margin=dict(l=0, r=0, t=30, b=0),
    font=dict(size=14),
)

fig_weekday.show()
fig_time.show()
fig_hist.show()

fig_weekday.write_image("fly_as_filed_weekday.png")
fig_time.write_image("fly_as_filed_time.png")
fig_hist.write_image("fly_as_filed_date.png")

110


***
### detect holding patterns

#### NOTE: Only run this if you have relatively large amout of memory, it's quite greedy
***

In [30]:
ort.set_default_logger_severity(3)

count_non_rekla = defaultdict(int)
count_rekla = defaultdict(int)

for dest in destintations:
    print(f"\n\n{dest.upper()}")

    count_failed_rekla = 0
    for flight in data.traf[f"routed_{dest}_rekla"]:
        holdings = flight.holding_pattern()
        count = 0
        try:
            for holding in holdings:
                count += 1
        except (ValueError, RecursionError):
            count_failed_rekla += 1
            # print(f"Flight {flight.flight_id} cannot be processed")
            continue

        if count > 0:
            count_rekla[dest] += 1
    print(f"Failed for REKLA: {count_failed_rekla}")

    count_failed_non_rekla = 0
    for flight in data.traf[f"routed_{dest}_non_rekla"]:
        holdings = flight.holding_pattern()
        count = 0
        try:
            for holding in holdings:
                count += 1
        except (ValueError, RecursionError):
            count_failed_non_rekla += 1
            # print(f"Flight {flight.flight_id} cannot be processed")
            continue

        if count > 0:
            count_non_rekla[dest] += 1
    print(f"Failed for non-REKLA: {count_failed_non_rekla}")




EGLL
Failed for REKLA: 0
Failed for non-REKLA: 2


EGKK
Failed for REKLA: 0
Failed for non-REKLA: 0


EGLC
Failed for REKLA: 0
Failed for non-REKLA: 0


In [31]:
for dest in destintations:
    routed_reklas = f"routed_{dest}_rekla"
    routed_non_reklas = f"routed_{dest}_non_rekla"

    print(f"\n\n{dest.upper()}")
    n_rekla = len(data.traf[routed_reklas])
    n_rekla_holding = count_rekla[dest]
    print(
        f"{n_rekla_holding} out of {n_rekla} on REKLA had a holding, {n_rekla_holding / n_rekla * 100:.2f}%"
    )

    n_non_rekla = len(data.traf[routed_non_reklas])
    n_non_rekla_holding = count_non_rekla[dest]
    print(
        f"{n_non_rekla_holding} out of {n_non_rekla} not on REKLA had a holding, {n_non_rekla_holding / n_non_rekla * 100:.2f}%"
    )



EGLL
42 out of 110 on REKLA had a holding, 38.18%
689 out of 2329 not on REKLA had a holding, 29.58%


EGKK
3 out of 26 on REKLA had a holding, 11.54%
76 out of 559 not on REKLA had a holding, 13.60%


EGLC
2 out of 22 on REKLA had a holding, 9.09%
13 out of 785 not on REKLA had a holding, 1.66%
